# Feature Engineering - F1 Podium Prediction

Membuat fitur untuk model prediksi podium sesuai rencana (Section 7).

## Fitur yang dibuat:
1. **Qualifying Features**: qualy_position, grid_effective, qual_delta, reached_q2/q3
2. **Driver Form**: rolling avg finish, points, podium_rate, win_rate, dnf_rate, grid_gain
3. **Constructor Form**: team points avg, team podium rate, team dnf rate
4. **Shifted Standings**: posisi dan points sebelum race
5. **Circuit History**: riwayat pembalap dan konstruktor di sirkuit yang sama
6. **Season Progress**: persentase musim telah berlalu
7. **Reliability Features**: mechanical_dnf_rate, crash_rate

**Semua rolling feature wajib menggunakan shift(1) untuk mencegah data leakage.**

In [41]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/'

In [42]:
# Load semua data
results = pd.read_csv(DATA_PATH + 'results.csv', low_memory=False)
races = pd.read_csv(DATA_PATH + 'races.csv', low_memory=False)
qualifying = pd.read_csv(DATA_PATH + 'qualifying.csv', low_memory=False)
drivers = pd.read_csv(DATA_PATH + 'drivers.csv', low_memory=False)
constructors = pd.read_csv(DATA_PATH + 'constructors.csv', low_memory=False)
circuits = pd.read_csv(DATA_PATH + 'circuits.csv', low_memory=False)
driver_standings = pd.read_csv(DATA_PATH + 'driver_standings.csv', low_memory=False)
constructor_standings = pd.read_csv(DATA_PATH + 'constructor_standings.csv', low_memory=False)
sprint_results = pd.read_csv(DATA_PATH + 'sprint_results.csv', low_memory=False)
status = pd.read_csv(DATA_PATH + 'status.csv', low_memory=False)

print(f'results: {results.shape}')
print(f'races: {races.shape}')
print(f'qualifying: {qualifying.shape}')
print(f'drivers: {drivers.shape}')
print(f'constructors: {constructors.shape}')
print(f'circuits: {circuits.shape}')
print(f'driver_standings: {driver_standings.shape}')
print(f'constructor_standings: {constructor_standings.shape}')
print(f'sprint_results: {sprint_results.shape}')
print(f'status: {status.shape}')

results: (27436, 18)
races: (1171, 18)
qualifying: (11168, 9)
drivers: (865, 9)
constructors: (214, 5)
circuits: (78, 9)
driver_standings: (35559, 7)
constructor_standings: (13730, 7)
sprint_results: (568, 17)
status: (140, 2)


## Data Cleaning

Konversi tipe data dan bersihkan nilai backslash-N.

In [43]:
def clean_numeric(series):
    return pd.to_numeric(series.replace(r'\N', np.nan, regex=False), errors='coerce')

# Cleaning results
results['grid'] = clean_numeric(results['grid'])
results['positionOrder'] = clean_numeric(results['positionOrder'])
results['points'] = clean_numeric(results['points'])
results['laps'] = clean_numeric(results['laps'])
results['position'] = clean_numeric(results['position'])

# Cleaning races
races['date'] = pd.to_datetime(races['date'])

# Cleaning qualifying
qualifying = qualifying.replace(r'\N', np.nan, regex=False)
qualifying['position'] = clean_numeric(qualifying['position'])
qualifying = qualifying.rename(columns={'position': 'qualy_position'})

# Cleaning standings
driver_standings['position'] = clean_numeric(driver_standings['position'])
driver_standings['points'] = clean_numeric(driver_standings['points'])
constructor_standings['position'] = clean_numeric(constructor_standings['position'])
constructor_standings['points'] = clean_numeric(constructor_standings['points'])

# Cleaning sprint
sprint_results['grid'] = clean_numeric(sprint_results['grid'])
sprint_results['positionOrder'] = clean_numeric(sprint_results['positionOrder'])
sprint_results['position'] = clean_numeric(sprint_results['position'])
sprint_results['points'] = clean_numeric(sprint_results['points'])

print('Data cleaning selesai.')

Data cleaning selesai.


## 1. Master Driver-Race Table

Buat tabel dasar: satu baris per driver per race.

In [44]:
# Merge results + races
df = results.merge(
    races[['raceId', 'year', 'round', 'circuitId', 'date', 'name']],
    on='raceId', how='left'
)

# Merge drivers
df = df.merge(drivers[['driverId', 'driverRef', 'code', 'forename', 'surname']], on='driverId', how='left')

# Merge constructors
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_team'))
df = df.rename(columns={'name': 'race_name', 'name_team': 'team'})

# Sort by date
df = df.sort_values(['date', 'raceId', 'positionOrder']).reset_index(drop=True)

# Target
df['is_podium'] = (df['positionOrder'] <= 3).astype(int)

print(f'Master table: {df.shape}')
print(f'Tahun: {df["year"].min()} - {df["year"].max()}')
print(f'Total race unik: {df["raceId"].nunique()}')

Master table: (27436, 29)
Tahun: 1950 - 2026
Total race unik: 1158


### Koreksi Starting Grid

Grid 0 (pit lane start) -> treat sebagai posisi terakhir.

In [45]:
# Grid efektif
field_size = df.groupby('raceId')['grid'].max()
df['field_size'] = df['raceId'].map(field_size)
df['grid_effective'] = df['grid'].fillna(df['field_size'] + 1)
df.loc[df['grid'] == 0, 'grid_effective'] = df['field_size'] + 1

print(f'Grid 0 count: {(df["grid"] == 0).sum()}')
print(f'Grid NaN count: {df["grid"].isna().sum()}')

Grid 0 count: 1638
Grid NaN count: 20


## 2. Qualifying Features

- qualy_position
- reached_q2, reached_q3
- qual_gap_to_pole (delta time dari pole dalam detik)

In [46]:
# Konversi waktu qualifying ke detik
def qual_time_to_seconds(time_str):
    if pd.isna(time_str):
        return np.nan
    try:
        parts = str(time_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        return float(time_str)
    except:
        return np.nan

for col in ['q1', 'q2', 'q3']:
    qualifying[f'{col}_sec'] = qualifying[col].apply(qual_time_to_seconds)

# Best qualifying time per driver per race
qualifying['best_qual_sec'] = qualifying[['q1_sec', 'q2_sec', 'q3_sec']].min(axis=1)

# Reached Q2, Q3
qualifying['reached_q2'] = qualifying['q2'].notna().astype(int)
qualifying['reached_q3'] = qualifying['q3'].notna().astype(int)

# Gap to pole per sesi qualifying
pole_time = qualifying.groupby('raceId')['best_qual_sec'].transform('min')
qualifying['qual_gap_to_pole'] = qualifying['best_qual_sec'] - pole_time

qualifying[['raceId', 'driverId', 'qualy_position', 'best_qual_sec', 'qual_gap_to_pole', 'reached_q2', 'reached_q3']].head()

,raceId,driverId,qualy_position,best_qual_sec,qual_gap_to_pole,reached_q2,reached_q3
0,18,1,1,85.187,0.000,1,1
1,18,9,2,85.315,0.128,1,1
2,18,5,3,85.452,0.265,1,1
3,18,13,4,85.691,0.504,1,1
4,18,2,5,85.518,0.331,1,1


In [47]:
# Merge qualifying features
qual_features = qualifying[[
    'raceId', 'driverId', 'qualy_position', 'best_qual_sec',
    'qual_gap_to_pole', 'reached_q2', 'reached_q3'
]]

df = df.merge(qual_features, on=['raceId', 'driverId'], how='left')

print(f'Setelah merge qualifying: {df.shape}')
print(f'qualy_position missing: {df["qualy_position"].isna().sum()}')

Setelah merge qualifying: (27436, 36)
qualy_position missing: 16269


## 3. Driver Form Features

Rolling window features dengan shift(1) untuk cegah leakage.

- driver_finish_avg_3, _5, _10
- driver_points_avg_3, _5, _10
- driver_podium_rate_5, _10
- driver_win_rate_10
- driver_dnf_rate_5, _10
- driver_grid_gain_avg_5
- driver_prev_finish, driver_prev_qualy

In [48]:
# Urutkan per driver berdasarkan waktu
df = df.sort_values(['driverId', 'date']).reset_index(drop=True)

# Fungsi rolling feature dengan shift(1) via transform (pandas 2.x compatible)
def rolling_shifted_transform(df_data, group_col, target_col, window):
    """Hitung rolling mean dengan shift(1) per group, return dengan index cocok."""
    return (
        df_data.groupby(group_col)[target_col]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# Driver finish position average (positionOrder closer to 1 = better)
df['driver_finish_avg_3'] = rolling_shifted_transform(df, 'driverId', 'positionOrder', 3)
df['driver_finish_avg_5'] = rolling_shifted_transform(df, 'driverId', 'positionOrder', 5)
df['driver_finish_avg_10'] = rolling_shifted_transform(df, 'driverId', 'positionOrder', 10)

print('Driver finish averages selesai.')

Driver finish averages selesai.


In [49]:
# Driver points average
df['driver_points_avg_3'] = rolling_shifted_transform(df, 'driverId', 'points', 3)
df['driver_points_avg_5'] = rolling_shifted_transform(df, 'driverId', 'points', 5)
df['driver_points_avg_10'] = rolling_shifted_transform(df, 'driverId', 'points', 10)

print('Driver points averages selesai.')

Driver points averages selesai.


In [50]:
# Driver podium rate
df['driver_podium_rate_5'] = rolling_shifted_transform(df, 'driverId', 'is_podium', 5)
df['driver_podium_rate_10'] = rolling_shifted_transform(df, 'driverId', 'is_podium', 10)

# Driver win rate
df['is_win'] = (df['positionOrder'] == 1).astype(int)
df['driver_win_rate_10'] = rolling_shifted_transform(df, 'driverId', 'is_win', 10)

print('Driver podium & win rates selesai.')

Driver podium & win rates selesai.


In [51]:
# Driver DNF rate
# statusId = 1 berarti finis normal, lainnya = DNF atau masalah
df['is_finished'] = (df['statusId'] == 1).astype(int)
df['is_dnf'] = (~df['is_finished'].astype(bool)).astype(int)

df['driver_dnf_rate_5'] = rolling_shifted_transform(df, 'driverId', 'is_dnf', 5)
df['driver_dnf_rate_10'] = rolling_shifted_transform(df, 'driverId', 'is_dnf', 10)

print('Driver DNF rates selesai.')

Driver DNF rates selesai.


In [52]:
# Grid gain: positif = naik posisi dari start ke finish
df['grid_gain'] = df['grid_effective'] - df['positionOrder']
df['driver_grid_gain_avg_5'] = rolling_shifted_transform(df, 'driverId', 'grid_gain', 5)

# Driver previous race finish (lag 1)
df['driver_prev_finish'] = df.groupby('driverId')['positionOrder'].shift(1)

# Driver previous race qualifying
df['driver_prev_qualy'] = df.groupby('driverId')['qualy_position'].shift(1)

print('Driver grid gain & prev features selesai.')

Driver grid gain & prev features selesai.


## 4. Constructor Form Features

- team_points_avg_5, _10
- team_best_finish_avg_5, _10
- team_podium_rate_10
- team_win_rate_10
- team_dnf_rate_5, _10
- constructor_prev_position, constructor_prev_points

In [53]:
# Urutkan per konstruktor
df = df.sort_values(['constructorId', 'date']).reset_index(drop=True)

# Team points
df['team_points_avg_5'] = rolling_shifted_transform(df, 'constructorId', 'points', 5)
df['team_points_avg_10'] = rolling_shifted_transform(df, 'constructorId', 'points', 10)

# Team podium rate
df['team_podium_rate_10'] = rolling_shifted_transform(df, 'constructorId', 'is_podium', 10)

# Team win rate
df['team_win_rate_10'] = rolling_shifted_transform(df, 'constructorId', 'is_win', 10)

# Team DNF rate
df['team_dnf_rate_5'] = rolling_shifted_transform(df, 'constructorId', 'is_dnf', 5)
df['team_dnf_rate_10'] = rolling_shifted_transform(df, 'constructorId', 'is_dnf', 10)

# Constructor previous position and points
df['constructor_prev_points'] = df.groupby('constructorId')['points'].shift(1)
df['constructor_prev_position'] = df.groupby('constructorId')['positionOrder'].shift(1)

print('Constructor form features selesai.')

Constructor form features selesai.


## 5. Shifted Standings

Posisi klasemen SEBELUM race (bukan setelah).
- driver_prev_standing_position
- driver_prev_standing_points
- driver_points_gap_to_leader
- constructor_prev_standing_position
- constructor_prev_standing_points

In [54]:
# Driver standings: shift(1) per driver
driver_standings_sorted = driver_standings.sort_values(['driverId', 'raceId']).reset_index(drop=True)
driver_standings_sorted['prev_standing_pos'] = driver_standings_sorted.groupby('driverId')['position'].shift(1)
driver_standings_sorted['prev_standing_points'] = driver_standings_sorted.groupby('driverId')['points'].shift(1)

# Gap to leader: cari poin tertinggi per race, lalu hitung selisih
max_points_per_race_standings = driver_standings_sorted.groupby('raceId')['points'].transform('max')
driver_standings_sorted['points_gap_to_leader'] = max_points_per_race_standings - driver_standings_sorted['points']

# Shift juga gap to leader
driver_standings_sorted['prev_points_gap_to_leader'] = driver_standings_sorted.groupby('driverId')['points_gap_to_leader'].shift(1)

# Merge
df = df.merge(
    driver_standings_sorted[['raceId', 'driverId', 'prev_standing_pos', 'prev_standing_points', 'prev_points_gap_to_leader']],
    on=['raceId', 'driverId'],
    how='left'
)

# Constructor standings: shift(1) per constructor
constructor_standings_sorted = constructor_standings.sort_values(['constructorId', 'raceId']).reset_index(drop=True)
constructor_standings_sorted['prev_constructor_pos'] = constructor_standings_sorted.groupby('constructorId')['position'].shift(1)
constructor_standings_sorted['prev_constructor_points'] = constructor_standings_sorted.groupby('constructorId')['points'].shift(1)

# Max constructor points per race
max_cons_points = constructor_standings_sorted.groupby('raceId')['points'].transform('max')
constructor_standings_sorted['cons_points_gap_to_leader'] = max_cons_points - constructor_standings_sorted['points']
constructor_standings_sorted['prev_cons_gap_to_leader'] = constructor_standings_sorted.groupby('constructorId')['cons_points_gap_to_leader'].shift(1)

# Merge
df = df.merge(
    constructor_standings_sorted[
        ['raceId', 'constructorId', 'prev_constructor_pos', 'prev_constructor_points', 'prev_cons_gap_to_leader']
    ],
    on=['raceId', 'constructorId'],
    how='left'
)

# Fill NaN untuk race pertama musim
for col in ['prev_standing_pos', 'prev_standing_points', 'prev_constructor_pos', 'prev_constructor_points']:
    df[col] = df[col].fillna(0)

print('Shifted standings features selesai.')

Shifted standings features selesai.


## 6. Circuit History

Riwayat pembalap di sirkuit yang sama, hanya dari race sebelumnya.

In [55]:
# Sort global by date
df = df.sort_values(['date', 'raceId', 'positionOrder']).reset_index(drop=True)

# Hitung statistik per driver + circuit
def create_circuit_history(df_data, group_cols, prefix):
    """
    Buat circuit history features per group.
    group_cols: ['driverId', 'circuitId'] atau ['constructorId', 'circuitId']
    """
    # Sort untuk akumulasi
    df_data = df_data.sort_values(group_cols + ['date']).reset_index(drop=True)
    
    # Rolling count of races at this circuit
    df_data[f'{prefix}_circuit_races'] = df_data.groupby(group_cols).cumcount()
    
    # Average finish at this circuit (expanding, shift untuk cegah leakage)
    # reset_index diperlukan karena .apply() menghasilkan MultiIndex
    n_levels = len(group_cols)
    df_data[f'{prefix}_circuit_avg_finish'] = (
        df_data.groupby(group_cols)['positionOrder']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).mean())
        .reset_index(level=list(range(n_levels)), drop=True)
    )
    
    # Podium rate at this circuit
    df_data[f'{prefix}_circuit_podium_rate'] = (
        df_data.groupby(group_cols)['is_podium']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).mean())
        .reset_index(level=list(range(n_levels)), drop=True)
    )
    
    # Best finish at this circuit
    df_data[f'{prefix}_circuit_best_finish'] = (
        df_data.groupby(group_cols)['positionOrder']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).min())
        .reset_index(level=list(range(n_levels)), drop=True)
    )
    
    return df_data

# Driver circuit history
df = create_circuit_history(df, ['driverId', 'circuitId'], 'driver')

# Constructor circuit history
df = create_circuit_history(df, ['constructorId', 'circuitId'], 'team')

# Fill NaN untuk first visit
for col in ['driver_circuit_avg_finish', 'driver_circuit_podium_rate', 'driver_circuit_best_finish',
            'team_circuit_avg_finish', 'team_circuit_podium_rate', 'team_circuit_best_finish']:
    df[col] = df[col].fillna(df[col].median() if col in ['driver_circuit_avg_finish', 'team_circuit_avg_finish'] else 0)

print('Circuit history features selesai.')

Circuit history features selesai.


## 7. Sprint Features

Untuk sprint weekend: sprint_grid, sprint_finish, sprint_points.

In [56]:
# Deteksi sprint weekend
sprint_races = sprint_results['raceId'].unique()
df['has_sprint'] = df['raceId'].isin(sprint_races).astype(int)

# Merge sprint results
sprint_features = sprint_results[['raceId', 'driverId', 'grid', 'positionOrder', 'points']].copy()
sprint_features = sprint_features.rename(columns={
    'grid': 'sprint_grid',
    'positionOrder': 'sprint_finish',
    'points': 'sprint_points'
})

df = df.merge(sprint_features, on=['raceId', 'driverId'], how='left')

# Isi NaN untuk non-sprint weekend
df['sprint_grid'] = df['sprint_grid'].fillna(0)
df['sprint_finish'] = df['sprint_finish'].fillna(0)
df['sprint_points'] = df['sprint_points'].fillna(0)

print(f'Sprint features selesai. Sprint weekends: {df["has_sprint"].sum()}')

Sprint features selesai. Sprint weekends: 568


## 8. Season Progress & Additional Features

In [57]:
# Season progress
total_rounds = df.groupby('year')['round'].max().to_dict()
df['total_rounds_in_season'] = df['year'].map(total_rounds)
df['season_progress'] = df['round'] / df['total_rounds_in_season']

# Field size (jumlah pembalap yang start)
field_size_per_race = df.groupby('raceId')['grid_effective'].count()
df['race_field_size'] = df['raceId'].map(field_size_per_race)

print('Season progress selesai.')

Season progress selesai.


In [58]:
# Kategorisasi reliability berdasarkan statusId

# Finished = 1, Mechanical = engine/gearbox/hydraulics/electrical, Crash = accident/collision
mechanical_statuses = [5, 6, 7, 8, 9, 10, 21, 22, 23, 24, 25, 26, 30, 32, 34, 36, 37, 38, 39, 40,
                       43, 44, 46, 47, 48, 51, 56, 91, 94, 95, 98, 99, 101, 102, 103, 105, 106, 108,
                       109, 110, 121, 126, 129, 131, 132, 135, 136, 141, 142]
crash_statuses = [3, 4, 20, 41, 65, 66, 130, 137, 138]

df['is_mechanical_dnf'] = df['statusId'].isin(mechanical_statuses).astype(int)
df['is_crash'] = df['statusId'].isin(crash_statuses).astype(int)

print(f'Mechanical DNF count: {df["is_mechanical_dnf"].sum()}')
print(f'Crash count: {df["is_crash"].sum()}')

Mechanical DNF count: 5924
Crash count: 2842


## 9. Reliability Features

Kategorikan statusId menjadi: Finished, Mechanical DNF, Crash, Disqualified, Other.

In [59]:
# Reliability rates via transform
df['driver_mechanical_dnf_rate_5'] = (
    df.groupby('driverId')['is_mechanical_dnf']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df['driver_crash_rate_5'] = (
    df.groupby('driverId')['is_crash']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Team rates
df['team_mechanical_dnf_rate_5'] = (
    df.groupby('constructorId')['is_mechanical_dnf']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print('Reliability features selesai.')

Reliability features selesai.


## 10. Teammate Comparison

Bandingkan performa driver dengan rekan setim di race yang sama.

In [60]:
# Cari teammate: driver lain di race yang sama dengan constructor yang sama
# Kita bisa lakukan dengan groupby raceId + constructorId

# Rata-rata finish position teammate per race
team_finish_mean = df.groupby(['raceId', 'constructorId'])['positionOrder'].transform('mean')
df['finish_gap_to_teammate_avg'] = df['positionOrder'] - team_finish_mean

# Rata-rata qualy position teammate
team_qualy_mean = df.groupby(['raceId', 'constructorId'])['qualy_position'].transform('mean')
df['qual_gap_to_teammate'] = df['qualy_position'] - team_qualy_mean

# Rolling qualifying win rate vs teammate
# Shifted: apakah di race sebelumnya qualy position lebih baik dari rata-rata tim?
df['qual_better_than_team_avg'] = (df['qualy_position'] < team_qualy_mean).astype(int)
df['qual_win_rate_vs_teammate_5'] = df.groupby('driverId')['qual_better_than_team_avg'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

# Rolling race win rate vs teammate
df['finish_better_than_team_avg'] = (df['positionOrder'] < team_finish_mean).astype(int)
df['race_win_rate_vs_teammate_5'] = df.groupby('driverId')['finish_better_than_team_avg'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

print('Teammate comparison features selesai.')

Teammate comparison features selesai.


## 11. Final Dataset

Filter periode 2014+ dan simpan dataset final.

In [61]:
# Filter era modern (2014+)
df_modern = df[df['year'] >= 2014].copy()

# Daftar fitur yang akan digunakan
feature_columns = [
    # Identitas
    'raceId', 'driverId', 'constructorId', 'year', 'round',
    'circuitId', 'date', 'race_name', 'driverRef', 'team',
    
    # Target
    'is_podium', 'positionOrder',
    
    # Qualifying
    'qualy_position', 'grid_effective', 'qual_gap_to_pole',
    'reached_q2', 'reached_q3',
    
    # Driver form
    'driver_finish_avg_3', 'driver_finish_avg_5', 'driver_finish_avg_10',
    'driver_points_avg_3', 'driver_points_avg_5', 'driver_points_avg_10',
    'driver_podium_rate_5', 'driver_podium_rate_10',
    'driver_win_rate_10',
    'driver_dnf_rate_5', 'driver_dnf_rate_10',
    'driver_grid_gain_avg_5',
    'driver_prev_finish', 'driver_prev_qualy',
    
    # Constructor form
    'team_points_avg_5', 'team_points_avg_10',
    'team_podium_rate_10', 'team_win_rate_10',
    'team_dnf_rate_5', 'team_dnf_rate_10',
    'constructor_prev_points', 'constructor_prev_position',
    
    # Standings
    'prev_standing_pos', 'prev_standing_points',
    'prev_points_gap_to_leader',
    'prev_constructor_pos', 'prev_constructor_points',
    'prev_cons_gap_to_leader',
    
    # Circuit history
    'driver_circuit_races', 'driver_circuit_avg_finish',
    'driver_circuit_podium_rate', 'driver_circuit_best_finish',
    'team_circuit_races', 'team_circuit_avg_finish',
    'team_circuit_podium_rate', 'team_circuit_best_finish',
    
    # Sprint
    'has_sprint', 'sprint_grid', 'sprint_finish', 'sprint_points',
    
    # Season
    'season_progress', 'race_field_size',
    
    # Reliability
    'driver_mechanical_dnf_rate_5', 'driver_crash_rate_5',
    'team_mechanical_dnf_rate_5',
    
    # Teammate comparison
    'qual_gap_to_teammate', 'finish_gap_to_teammate_avg',
    'qual_win_rate_vs_teammate_5', 'race_win_rate_vs_teammate_5',
]

# Cek kolom yang ada
available_features = [c for c in feature_columns if c in df_modern.columns]
missing_features = [c for c in feature_columns if c not in df_modern.columns]

print(f'Fitur tersedia: {len(available_features)} dari {len(feature_columns)}')
if missing_features:
    print(f'Fitur tidak ditemukan: {missing_features}')

df_final = df_modern[available_features].copy()
print(f'\nFinal dataset shape: {df_final.shape}')
print(f'Tahun: {df_final["year"].min()} - {df_final["year"].max()}')

Fitur tersedia: 66 dari 66

Final dataset shape: (5303, 66)
Tahun: 2014 - 2026


In [62]:
# Simpan dataset
df_final.to_parquet('../data/processed/model_dataset.parquet', index=False)
print('Dataset saved to data/processed/model_dataset.parquet')

Dataset saved to data/processed/model_dataset.parquet


In [63]:
# Statistik dasar dataset final
print(f'Shape: {df_final.shape}')
print(f'Unique races: {df_final["raceId"].nunique()}')
print(f'Unique drivers: {df_final["driverId"].nunique()}')
print(f'Unique constructors: {df_final["constructorId"].nunique()}')
print(f'\nMissing values per kolom:')
missing_stats = df_final.isna().sum()
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
display(missing_stats)

Shape: (5303, 66)
Unique races: 261
Unique drivers: 63
Unique constructors: 22

Missing values per kolom:


qual_gap_to_pole                76
driver_prev_qualy               62
driver_mechanical_dnf_rate_5    45
driver_crash_rate_5             45
race_win_rate_vs_teammate_5     45
qual_win_rate_vs_teammate_5     45
driver_grid_gain_avg_5          42
prev_points_gap_to_leader       42
driver_finish_avg_5             42
driver_points_avg_3             42
driver_finish_avg_3             42
driver_dnf_rate_5               42
driver_dnf_rate_10              42
driver_win_rate_10              42
driver_podium_rate_10           42
driver_points_avg_5             42
driver_points_avg_10            42
driver_podium_rate_5            42
driver_finish_avg_10            42
driver_prev_finish              42
qualy_position                  20
grid_effective                  20
reached_q3                      20
reached_q2                      20
qual_gap_to_teammate            20
prev_cons_gap_to_leader         16
team_mechanical_dnf_rate_5      10
constructor_prev_position        8
constructor_prev_poi